# Shared RRF data for fine-tuning

Builds the train/val/test files. 

70/10/20 split of the 1,000 PubMedQA questions (seed 420). 

Each question gets its top 5 abstracts from RRF (BM25 + PubMedBERT) over all 1,000 abstracts. 

Matches the split and retrieval in Brent's SFT notebook.

In [1]:
import pandas as pd
import bm25s
from txtai import Embeddings

/Users/andrew/Documents/Berkeley/MIDS/DS266/Final Project/DS266_RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1122 19:30:51.990000 48936 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Load and split

In [2]:
# load the 1,000 questions
data = pd.read_json("pubmedqa/data/ori_pqal.json").T

# join each abstract's sections into one string
data["CONTEXTS_combined"] = ["".join(entry) for entry in data["CONTEXTS"]]

# shuffle and 70/10/20 split
data_shuffled = data.sample(frac=1, random_state=420)
train = data_shuffled.iloc[:700]
val = data_shuffled.iloc[700:800]
test = data_shuffled.iloc[800:]

print(len(train), len(val), len(test))
print("train:", train.index[:5].tolist())
print("test:", test.index[:5].tolist())

700 100 200
train: [19854401, 21481154, 15388567, 24298614, 22349635]
test: [26104852, 10759659, 26200172, 23348330, 17076091]


## Build search

In [3]:
# lowercase every abstract
all_contexts = [context.lower() for context in data["CONTEXTS_combined"]]
all_ids = data.index.to_numpy()

# keyword search (bm25) over all 1,000 abstracts
all_tokens = bm25s.tokenize(all_contexts, stopwords="en")
sparse_search = bm25s.BM25(corpus=all_ids)
sparse_search.index(all_tokens)

# meaning search (pubmedbert) over all 1,000 abstracts
all_str_ids = data.index.astype(str).tolist()
dense_search = Embeddings(path="neuml/pubmedbert-base-embeddings")
dense_search.index(list(zip(all_str_ids, all_contexts)))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7516.27it/s]


## Retrieve top abstracts

In [4]:
# find the top abstracts for a question using keyword and meaning search
def get_top_abstracts(question, num_search=10, num_keep=5):
    # keyword search
    question_tokens = bm25s.tokenize([question.lower()], stopwords="en")
    sparse_results, _ = sparse_search.retrieve(question_tokens, k=num_search)
    sparse_ids = [str(doc_id) for doc_id in sparse_results[0]]

    # meaning search
    dense_results = dense_search.search(question, num_search)
    dense_ids = [str(result[0]) for result in dense_results]

    # give each abstract points based on its rank in each search
    points = {}
    for ranking in [sparse_ids, dense_ids]:
        for rank in range(len(ranking)):
            doc_id = ranking[rank]
            if doc_id not in points:
                points[doc_id] = 0
            points[doc_id] += 1 / (60 + rank + 1)

    # keep the abstracts with the most points
    sorted_ids = sorted(points, key=points.get, reverse=True)

    top_ids = []
    for doc_id in sorted_ids[:num_keep]:
        top_ids.append(int(doc_id))

    top_abstracts = data.loc[top_ids, "CONTEXTS_combined"].tolist()
    return top_ids, top_abstracts


# try it on the first training question
top_ids, top_abstracts = get_top_abstracts(train["QUESTION"].iloc[0])
print(top_ids)

[19854401, 26471488, 18243752, 11034241, 15631914]


## Build the files

In [5]:
# find the top abstracts for every question in a split
def build_records(split):
    records = []
    for pubmed_id, row in split.iterrows():
        top_ids, top_abstracts = get_top_abstracts(row["QUESTION"])

        records.append({
            "pubid": int(pubmed_id),
            "question": row["QUESTION"],
            "retrieved_ids": top_ids,
            "context": {"contexts": top_abstracts},
            "long_answer": row["LONG_ANSWER"],
            "final_decision": row["final_decision"],
        })

    return pd.DataFrame(records)


train_rrf = build_records(train)
val_rrf = build_records(val)
test_rrf = build_records(test)

print(len(train_rrf), len(val_rrf), len(test_rrf))

700 100 200


## Save

In [6]:
# save each split as a jsonl file
train_rrf.to_json("pubmedqa/data/pubmedqa_train_rrf.jsonl", orient="records", lines=True)
val_rrf.to_json("pubmedqa/data/pubmedqa_val_rrf.jsonl", orient="records", lines=True)
test_rrf.to_json("pubmedqa/data/pubmedqa_test_rrf.jsonl", orient="records", lines=True)

## Check retrieval

In [7]:
# how often the correct abstract is in the top 5
def recall_at_5(split_rrf):
    found = 0
    for i in range(len(split_rrf)):
        correct_id = split_rrf["pubid"].iloc[i]
        retrieved_ids = split_rrf["retrieved_ids"].iloc[i]
        if correct_id in retrieved_ids:
            found += 1
    return found / len(split_rrf)


print("train:", recall_at_5(train_rrf))
print("val:", recall_at_5(val_rrf))
print("test:", recall_at_5(test_rrf))

train: 0.9928571428571429
val: 0.99
test: 1.0
